# Wasserstein KD, vertical and horizontal

This notebook runs the two Wasserstein branches, one per card.

| Branch | Loss for the narrow widths | Role |
|---|---|---|
| `c_wasserstein` | W(student, teacher) on the logits | WKD, vertical only |
| `d_wasserstein_pair` | plus lambda * W(s1, s2) | the contribution |

D adds a term between the two middle widths the sandwich rule already draws every step. Two students at different widths stand in no teacher-student relation, so the asymmetry of KL is wrong there on principle rather than merely worse, and nothing about the term needs a discrete grid.

Both are slower than A and B: the Sinkhorn solve runs three times per narrow width, and D defers backward so two graphs stay alive at once.

Set the accelerator to **GPU T4 x2**. Each branch gets a card of its own: two independent jobs pinned with `CUDA_VISIBLE_DEVICES`, so each sees one GPU and `num_gpus_per_job` is 1.

This is not data parallelism, and deliberately so. Splitting one batch over both cards would leave BN collecting running statistics from one card's half only, and would tie the two branches to a single failure: when the session times out both die at the same epoch. Independent jobs also give two clean per-branch logs rather than one interleaved one.


In [ ]:
# This notebook is pinned to these two branches, one per card.
BRANCHES = ['c_wasserstein', 'd_wasserstein_pair']

SMOKE_FIRST = True  # 6 steps per branch before committing the session

# Where the code comes from. Either push the branch and clone it, or attach
# the repository as a Kaggle dataset and leave REPO_URL empty.
REPO_URL = 'https://github.com/duyh80456-code/new-pruning.git'
REPO_BRANCH = 'nhan'

# To resume, add the previous run's output as an input and name its
# logs directory here.
RESUME_FROM = ''  # e.g. '/kaggle/input/wasserstein-kd-ab/logs'


In [ ]:
import os
import queue
import shutil
import subprocess
import sys
import threading
import time

import torch

print('torch', torch.__version__)
n_gpu = torch.cuda.device_count()
print('gpus', n_gpu)
for i in range(n_gpu):
    props = torch.cuda.get_device_properties(i)
    print('  {}: {}  {:.1f} GB'.format(
        i, props.name, props.total_memory / 1e9))

if n_gpu < len(BRANCHES):
    print('\n{} branches but {} gpu(s): they will run one after another.'
          .format(len(BRANCHES), n_gpu))

In [ ]:
WORK = '/kaggle/working'
CODE = os.path.join(WORK, 'new-pruning')

if REPO_URL:
    if not os.path.isdir(CODE):
        subprocess.run(
            ['git', 'clone', '-b', REPO_BRANCH, REPO_URL, CODE], check=True)
else:
    found = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train.py' in files and 'apps' in dirs:
            found = root
            break
    if found is None:
        raise SystemExit(
            'Set REPO_URL, or attach the repository as a Kaggle dataset.')
    if os.path.isdir(CODE):
        shutil.rmtree(CODE)
    shutil.copytree(found, CODE)
    print('copied code from', found)

os.chdir(CODE)
print('cwd', os.getcwd())
print(sorted(os.listdir('.')))

## Put CIFAR-100 in place, once, before anything is launched

Prefers an attached dataset and falls back to downloading. Two jobs
starting together would otherwise both find the folder missing and
unpack into it at the same time, so this has to happen here rather than
inside the training processes.

torchvision wants `<root>/cifar-100-python`, so the attached copy is
linked to that name under `data/`. The input mount is read only, which
is fine: nothing writes to it.


In [ ]:
# The dataset as attached. Leave as is to search /kaggle/input for it.
CIFAR_DIR = ('/kaggle/input/datasets/nlnk1607/cifar100/cifar-100-python')

TARGET = 'data/cifar-100-python'

if os.path.isdir(TARGET):
    print('already in place:', TARGET)
else:
    source = CIFAR_DIR if os.path.isdir(CIFAR_DIR) else None
    if source is None:
        for root, dirs, _ in os.walk('/kaggle/input'):
            if 'cifar-100-python' in dirs:
                source = os.path.join(root, 'cifar-100-python')
                break
    os.makedirs('data', exist_ok=True)
    if source:
        try:
            os.symlink(source, TARGET)
            print('linked', source)
        except OSError:
            shutil.copytree(source, TARGET)
            print('copied', source)
    else:
        print('no attached copy found, downloading')
        from torchvision import datasets
        datasets.CIFAR100(root='data', train=True, download=True)
        datasets.CIFAR100(root='data', train=False, download=True)

# torchvision reads train, test and meta from this folder; data/cifar100.py
# only downloads when it is absent, so both jobs now skip the download.
print(sorted(os.listdir(TARGET)))


## Check the loss before spending the session on it

The Sinkhorn potentials are solved without gradient and the gradient is
taken from them by the envelope theorem, which only holds at convergence.
An under-solved Sinkhorn gives a wrong gradient and raises nothing, so the
marginal violation is worth reading even when every check passes.

In [ ]:
!python tests/test_loss_ops.py

## Is there any geometry in the classifier?

The cost matrix is `d(i, j) = ||w_i - w_j||` over classifier rows, and the
whole logit-level argument rests on those distances differing. In high
dimensions random rows are all nearly equidistant, and a flat cost matrix
turns transport into a scaled total variation, which is just another
f-divergence.

A ratio near 1.3 means the argument has no content on this model. Around 3
or above means there is structure to exploit. Worth running again on a
trained checkpoint, where it decides something.

In [ ]:
probe = r'''
import os, sys, torch
sys.argv.append("app:apps/cifar100_c_wasserstein.yml")
from utils.loss_ops import class_cost_matrix

def spread(name, w):
    c = class_cost_matrix(w)
    off = c + torch.eye(c.size(0)) * 1e9
    near, far = off.min(1)[0].mean().item(), c.max(1)[0].mean().item()
    print("{:26} nearest {:.3f}  farthest {:.3f}  ratio {:.2f}".format(
        name, near, far, far / near))

spread("random init", torch.randn(100, 512))
for branch in sys.argv[1:]:
    ckpt = "logs/cifar100_{}/best_model.pt".format(branch)
    if os.path.exists(ckpt):
        state = torch.load(ckpt, map_location="cpu")["model"]
        key = [k for k in state if k.endswith("classifier.0.weight")][0]
        spread("trained " + branch, state[key])
'''


def probe_geometry():
    done = subprocess.run(
        [sys.executable, '-c', probe] + BRANCHES,
        capture_output=True, text=True)
    print(done.stdout or done.stderr)


probe_geometry()

In [ ]:
# Resume: copy a previous session's logs back into place. train.py picks up
# logs/<name>/latest_checkpoint.pt on its own and restores model, optimizer,
# epoch, best val and meters. The configs use cosine_decaying, whose rate is
# a function of the epoch rather than an accumulated subtraction, so a
# resumed run follows exactly the schedule an uninterrupted one would.
if RESUME_FROM:
    os.makedirs('logs', exist_ok=True)
    for name in os.listdir(RESUME_FROM):
        src = os.path.join(RESUME_FROM, name)
        if os.path.isdir(src):
            shutil.copytree(src, os.path.join('logs', name),
                            dirs_exist_ok=True)
    for root, _, files in os.walk('logs'):
        for f in files:
            path = os.path.join(root, f)
            print('{:52} {:>8.1f} MB'.format(
                path, os.path.getsize(path) / 1e6))
else:
    print('starting from scratch')

In [ ]:
def run_pinned(jobs, quiet_model_repr=True):
    """run one job per gpu and interleave their output

    jobs is a list of (label, config path). Each is given one card through
    CUDA_VISIBLE_DEVICES, so inside the process it is always cuda:0 and the
    two never contend for memory. Lines are tagged with the branch so the
    combined log stays readable.
    """
    lines = queue.Queue()
    procs = {}

    def pump(label, proc):
        for line in proc.stdout:
            lines.put((label, line.rstrip('\n')))
        proc.wait()
        lines.put((label, None))

    for index, (label, config) in enumerate(jobs):
        env = dict(os.environ)
        env['CUDA_VISIBLE_DEVICES'] = str(index % max(n_gpu, 1))
        proc = subprocess.Popen(
            [sys.executable, '-u', 'train.py', 'app:' + config],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, env=env)
        procs[label] = proc
        threading.Thread(target=pump, args=(label, proc),
                         daemon=True).start()
        print('[{}] started on gpu {} with {}'.format(
            label, env['CUDA_VISIBLE_DEVICES'], config), flush=True)

    started = time.time()
    remaining = len(jobs)
    while remaining:
        label, line = lines.get()
        if line is None:
            remaining -= 1
            print('[{}] exit code {} after {:.0f} min'.format(
                label, procs[label].returncode,
                (time.time() - started) / 60), flush=True)
            continue
        if quiet_model_repr and (
                line.startswith(('  ', ')', 'DataParallel', 'Model('))
                or not line.strip()):
            continue  # the model repr, printed once at startup
        print('[{}] {}'.format(label, line), flush=True)

    return {label: proc.returncode for label, proc in procs.items()}

In [ ]:
# Smoke run: 3 iterations an epoch, two epochs, four test widths. Catches a
# broken config or a shape error in a minute instead of at hour three.
if SMOKE_FIRST:
    codes = run_pinned(
        [(b, 'apps/smoke_{}.yml'.format(b)) for b in BRANCHES])
    failed = [b for b, code in codes.items() if code != 0]
    if failed:
        raise SystemExit(
            'smoke run failed for {}, do not start the real one'.format(
                failed))
    for b in BRANCHES:
        shutil.rmtree('logs/smoke_{}'.format(b), ignore_errors=True)
    print('\nsmoke ok')

In [ ]:
# The real run. Every epoch writes logs/<name>/latest_checkpoint.pt, so a
# session that times out loses at most one epoch.
codes = run_pinned(
    [(b, 'apps/cifar100_{}.yml'.format(b)) for b in BRANCHES])
print(codes)

## Results

The numbers that matter come after BN calibration, which recomputes the
post-statistics for every width in `width_mult_list_test`. Validation
accuracy during training is measured with training-time BN statistics, so
read it as a progress signal only.

One seed is not a result. The project's own specialization gap moved by
0.73 / 2.71 / 3.15 across three seeds at one width, so a gap of a few
tenths of a point between branches means nothing until sigma is known.

In [ ]:
probe_geometry()

out = os.path.join(WORK, 'logs')
for b in BRANCHES:
    log_dir = 'logs/cifar100_{}'.format(b)
    if not os.path.isdir(log_dir):
        print('{}: nothing written'.format(b))
        continue
    print('\n' + log_dir)
    for name in sorted(os.listdir(log_dir)):
        path = os.path.join(log_dir, name)
        print('  {:34} {:>8.1f} MB'.format(
            name, os.path.getsize(path) / 1e6))
    # keep the checkpoints as session output so the next run can resume
    shutil.copytree(log_dir, os.path.join(out, os.path.basename(log_dir)),
                    dirs_exist_ok=True)
print('\ncheckpoints copied to', out)